# Курсы ЦБ и оптимизация индикаторов

Ноутбук последовательно выполняет весь исследовательский pipeline:

1. загружает курсы ЦБ;
2. строит causal-признаки и targets;
3. задаёт пространства правил;
4. оптимизирует rule-based индикаторы через walk-forward;
5. проверяет лучшие фиксированные G0/W1-индикаторы на независимом holdout.


In [1]:
from datetime import date
from itertools import combinations, product
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.cbr_loader import CURRENCIES, load_cbr_history
from src.features import build_features
from src.indicator_optimization import optimize_indicator_library
from src.indicator_backtest import backtest_fixed_indicators
from src.indicators import IndicatorCandidate, create_indicator_rules
from src.market_data import build_daily_market_panel, market_snapshot
from src.outcomes import DEFAULT_HORIZONS, add_future_outcomes, outcome_columns
from src.target_evaluation import (
    evaluate_perfect_target_lift,
    evaluate_target_family_frequency,
)
from src.targets import build_targets, target_columns

START_DATE = date(2020, 1, 1)
END_DATE = date.today()
RECIPIENT_CURRENCIES = ("AMD", "KZT", "KGS", "TJS", "UZS")

# Проверяем несколько частот переобучения как полноценный параметр оптимизации.
FIRST_TEST_DATE = "2022-01-01"
TEST_MONTHS_OPTIONS = (3, 6, 12)
INDICATOR_MIN_SIGNALS_PER_WEEK = 2.0
ENSEMBLE_LIFT_THRESHOLDS = (1.3,)
FINAL_BACKTEST_START = pd.Timestamp("2025-01-01")
FINAL_BACKTEST_TARGET_FAMILIES = ("G0", "W1")
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "cbr"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


## 1. Данные ЦБ

Номиналы приводятся к рублям за одну единицу валюты. `available_at` консервативно установлен на 00:00 следующего календарного дня. Сохраняется только базовое хранилище курсов, потому что оно является входом всего исследования.


In [2]:
cbr_history = load_cbr_history(
    start_date=START_DATE,
    end_date=END_DATE,
    currencies=CURRENCIES,
    raw_dir=RAW_DIR,
)
assert cbr_history["normalized_rate"].gt(0).all()
assert cbr_history["publication_timestamp_is_proxy"].all()
assert (cbr_history["available_at"] == cbr_history["effective_date"] + pd.Timedelta(days=1)).all()
pd.testing.assert_series_equal(
    cbr_history["normalized_rate"],
    cbr_history["raw_rate"] / cbr_history["nominal"],
    check_names=False,
)

rates = (
    cbr_history.pivot(index="available_at", columns="currency", values="normalized_rate")
    .reindex(columns=list(CURRENCIES))
    .sort_index()
)
rates.columns.name = None
rates.index.name = "available_at"
assert rates.notna().all().all()
assert rates.index.is_unique and rates.index.is_monotonic_increasing

market_panel = build_daily_market_panel(rates)
latest_snapshot = market_snapshot(market_panel, as_of=market_panel["available_at"].max())
assert set(latest_snapshot["currency"]) == set(CURRENCIES)

cbr_history.to_parquet(PROCESSED_DIR / "cbr_history_temporal.parquet", index=False)
rates.to_parquet(PROCESSED_DIR / "cbr_rates.parquet")
rates.to_csv(PROCESSED_DIR / "cbr_rates.csv")

pd.DataFrame({
    "publication_dates": [len(rates)],
    "first_available_at": [rates.index.min()],
    "last_available_at": [rates.index.max()],
    "currencies": [", ".join(rates.columns)],
})


,publication_dates,first_available_at,last_available_at,currencies
0,1643,2020-01-02,2026-09-04,"AMD, KZT, KGS, TJS, UZS, USD, EUR, CNY"


## 2. Features и targets

Признаки используют только текущие и прошлые значения. Future outcomes нужны исключительно для разметки. Оптимизация выполняется только на реальных днях публикации курсов пяти целевых валют.


In [3]:
features = build_features(market_panel)
outcomes = add_future_outcomes(features, horizons=DEFAULT_HORIZONS)
dataset, target_registry = build_targets(outcomes, horizons=DEFAULT_HORIZONS)

assert set(target_registry["horizon"]) == set(DEFAULT_HORIZONS)
assert target_registry["name"].isin(dataset.columns).all()
assert len(target_columns(dataset)) == len(target_registry)
assert outcome_columns(dataset)

update_rows = dataset.loc[
    dataset["is_update_day"] & dataset["currency"].isin(RECIPIENT_CURRENCIES)
].copy()
target_prevalence = (
    update_rows[["currency", *target_registry["name"].tolist()]]
    .melt(id_vars="currency", var_name="target", value_name="value")
    .dropna(subset=["value"])
    .groupby(["currency", "target"], as_index=False)
    .agg(observations=("value", "size"), positive_rate=("value", "mean"))
    .merge(
        target_registry[["name", "scenario", "family", "horizon"]],
        left_on="target", right_on="name", how="left", validate="many_to_one",
    )
    .drop(columns="name")
)
target_prevalence.pivot_table(
    index=["scenario", "family", "target", "horizon"],
    columns="currency", values="positive_rate",
).sort_index()


currency                                                                           AMD  \
scenario       family target                                         horizon             
GOOD_NOW       G0     target_g0_exact_min_h10d                       10       0.042866   
                      target_g0_exact_min_h1d                        1        0.274223   
                      target_g0_exact_min_h20d                       20       0.020358   
                      target_g0_exact_min_h3d                        3        0.126297   
                      target_g0_exact_min_h5d                        5        0.087858   
               G1     target_g1_regret_le_100bps_h10d                10        0.58104   
                      target_g1_regret_le_100bps_h1d                 1         0.90134   
                      target_g1_regret_le_100bps_h20d                20       0.464088   
                      target_g1_regret_le_100bps_h3d                 3        0.803659   
                      target_g1_regret_le_100bps_h5d                 5        0.712805   
                      target_g1_regret_le_25bps_h10d                 10       0.359021   
                      target_g1_regret_le_25bps_h1d                  1        0.717418   
                      target_g1_regret_le_25bps_h20d                 20       0.279312   
                      target_g1_regret_le_25bps_h3d                  3        0.552439   
                      target_g1_regret_le_25bps_h5d                  5        0.462195   
                      target_g1_regret_le_50bps_h10d                 10       0.458716   
                      target_g1_regret_le_50bps_h1d                  1        0.811815   
                      target_g1_regret_le_50bps_h20d                 20       0.360344   
                      target_g1_regret_le_50bps_h3d                  3        0.665244   
                      target_g1_regret_le_50bps_h5d                  5        0.571951   
WINDOW_CLOSING W0     target_w0_deterioration_75bps_h10d             10       0.407951   
                      target_w0_deterioration_75bps_h1d              1        0.137028   
                      target_w0_deterioration_75bps_h20d             20       0.456722   
                      target_w0_deterioration_75bps_h3d              3        0.243293   
                      target_w0_deterioration_75bps_h5d              5        0.332317   
               W1     target_w1_lowpct_0p15_deterioration_75bps_h10d 10       0.080431   
                      target_w1_lowpct_0p15_deterioration_75bps_h1d  1        0.027743   
                      target_w1_lowpct_0p15_deterioration_75bps_h20d 20       0.089002   
                      target_w1_lowpct_0p15_deterioration_75bps_h3d  3        0.051136   
                      target_w1_lowpct_0p15_deterioration_75bps_h5d  5        0.064394   

currency                                                                           KGS  \
scenario       family target                                         horizon             
GOOD_NOW       G0     target_g0_exact_min_h10d                       10       0.039192   
                      target_g0_exact_min_h1d                        1         0.27727   
                      target_g0_exact_min_h20d                       20       0.021592   
                      target_g0_exact_min_h3d                        3        0.133618   
                      target_g0_exact_min_h5d                        5        0.084808   
               G1     target_g1_regret_le_100bps_h10d                10       0.551682   
                      target_g1_regret_le_100bps_h1d                 1        0.903167   
                      target_g1_regret_le_100bps_h20d                20       0.443831   
                      target_g1_regret_le_100bps_h3d                 3        0.794512   
                      target_g1_regret_le_100bps_h5d                 5        0.704268   
                      target_g1_regret_

## 3. Теоретический lift таргетов до обучения моделей

Для каждой полной конфигурации `currency × exact target × horizon` рассматривается идеальный предиктор, который подаёт сигнал ровно на положительных наблюдениях. Его precision равна 1, precision случайного входа равна доле положительного класса, поэтому `ideal_lift_vs_random = 1 / random_precision`. Это верхняя теоретическая граница lift для бинарной метрики таргета, а не ожидаемое качество будущей модели. `positive_count` нужно рассматривать вместе с lift: редкий таргет автоматически получает высокий теоретический lift.


In [4]:
target_ideal_lift = evaluate_perfect_target_lift(
    update_rows,
    target_registry=target_registry,
)

target_ideal_lift_pivot = target_ideal_lift.pivot_table(
    index=["scenario", "target_family", "target", "horizon"],
    columns="currency",
    values="ideal_lift_vs_random",
).sort_index()
target_ideal_lift_pivot


currency                                                                                   AMD  \
scenario       target_family target                                         horizon              
GOOD_NOW       G0            target_g0_exact_min_h10d                       10       23.328571   
                             target_g0_exact_min_h1d                        1         3.646667   
                             target_g0_exact_min_h20d                       20       49.121212   
                             target_g0_exact_min_h3d                        3         7.917874   
                             target_g0_exact_min_h5d                        5        11.381944   
               G1            target_g1_regret_le_100bps_h10d                10        1.721053   
                             target_g1_regret_le_100bps_h1d                 1         1.109459   
                             target_g1_regret_le_100bps_h20d                20        2.154762   
                             target_g1_regret_le_100bps_h3d                 3         1.244310   
                             target_g1_regret_le_100bps_h5d                 5         1.402908   
                             target_g1_regret_le_25bps_h10d                 10        2.785349   
                             target_g1_regret_le_25bps_h1d                  1         1.393888   
                             target_g1_regret_le_25bps_h20d                 20        3.580220   
                             target_g1_regret_le_25bps_h3d                  3         1.810155   
                             target_g1_regret_le_25bps_h5d                  5         2.163588   
                             target_g1_regret_le_50bps_h10d                 10        2.180000   
                             target_g1_regret_le_50bps_h1d                  1         1.231808   
                             target_g1_regret_le_50bps_h20d                 20        2.775128   
                             target_g1_regret_le_50bps_h3d                  3         1.503208   
                             target_g1_regret_le_50bps_h5d                  5         1.748401   
WINDOW_CLOSING W0            target_w0_deterioration_75bps_h10d             10        2.451274   
                             target_w0_deterioration_75bps_h1d              1         7.297778   
                             target_w0_deterioration_75bps_h20d             20        2.189516   
                             target_w0_deterioration_75bps_h3d              3         4.110276   
                             target_w0_deterioration_75bps_h5d              5         3.009174   
               W1            target_w1_lowpct_0p15_deterioration_75bps_h10d 10       12.433071   
                             target_w1_lowpct_0p15_deterioration_75bps_h1d  1        36.045455   
                             target_w1_lowpct_0p15_deterioration_75bps_h20d 20       11.235714   
                             target_w1_lowpct_0p15_deterioration_75bps_h3d  3        19.555556   
                             target_w1_lowpct_0p15_deterioration_75bps_h5d  5        15.529412   

currency                                                                                   KGS  \
scenario       target_family target                                         horizon              
GOOD_NOW       G0            target_g0_exact_min_h10d                       10       25.515625   
                             target_g0_exact_min_h1d                        1         3.606593   
                             target_g0_exact_min_h20d                       20       46.314286   
                             target_g0_exact_min_h3d                        3         7.484018   
                             target_g0_exact_min_h5d                        5        11.791367   
               G1            target_g1_regret_le_100bps_h10d                10        1.812639   
                             target_g1_regret_le_100bps_h1d                 1  

In [5]:
target_ideal_lift_pivot[target_ideal_lift_pivot[['AMD', 'KGS', 'KZT', 'TJS', 'UZS']]>1.3]

currency                                                                                   AMD  \
scenario       target_family target                                         horizon              
GOOD_NOW       G0            target_g0_exact_min_h10d                       10       23.328571   
                             target_g0_exact_min_h1d                        1         3.646667   
                             target_g0_exact_min_h20d                       20       49.121212   
                             target_g0_exact_min_h3d                        3         7.917874   
                             target_g0_exact_min_h5d                        5        11.381944   
               G1            target_g1_regret_le_100bps_h10d                10        1.721053   
                             target_g1_regret_le_100bps_h1d                 1              NaN   
                             target_g1_regret_le_100bps_h20d                20        2.154762   
                             target_g1_regret_le_100bps_h3d                 3              NaN   
                             target_g1_regret_le_100bps_h5d                 5         1.402908   
                             target_g1_regret_le_25bps_h10d                 10        2.785349   
                             target_g1_regret_le_25bps_h1d                  1         1.393888   
                             target_g1_regret_le_25bps_h20d                 20        3.580220   
                             target_g1_regret_le_25bps_h3d                  3         1.810155   
                             target_g1_regret_le_25bps_h5d                  5         2.163588   
                             target_g1_regret_le_50bps_h10d                 10        2.180000   
                             target_g1_regret_le_50bps_h1d                  1              NaN   
                             target_g1_regret_le_50bps_h20d                 20        2.775128   
                             target_g1_regret_le_50bps_h3d                  3         1.503208   
                             target_g1_regret_le_50bps_h5d                  5         1.748401   
WINDOW_CLOSING W0            target_w0_deterioration_75bps_h10d             10        2.451274   
                             target_w0_deterioration_75bps_h1d              1         7.297778   
                             target_w0_deterioration_75bps_h20d             20        2.189516   
                             target_w0_deterioration_75bps_h3d              3         4.110276   
                             target_w0_deterioration_75bps_h5d              5         3.009174   
               W1            target_w1_lowpct_0p15_deterioration_75bps_h10d 10       12.433071   
                             target_w1_lowpct_0p15_deterioration_75bps_h1d  1        36.045455   
                             target_w1_lowpct_0p15_deterioration_75bps_h20d 20       11.235714   
                             target_w1_lowpct_0p15_deterioration_75bps_h3d  3        19.555556   
                             target_w1_lowpct_0p15_deterioration_75bps_h5d  5        15.529412   

currency                                                                                   KGS  \
scenario       target_family target                                         horizon              
GOOD_NOW       G0            target_g0_exact_min_h10d                       10       25.515625   
                             target_g0_exact_min_h1d                        1         3.606593   
                             target_g0_exact_min_h20d                       20       46.314286   
                             target_g0_exact_min_h3d                        3         7.484018   
                             target_g0_exact_min_h5d                        5        11.791367   
               G1            target_g1_regret_le_100bps_h10d                10        1.812639   
                             target_g1_regret_le_100bps_h1d                 1  

### 3.1. Частота уникальных положительных дат по семействам

Для каждой валюты варианты одного `target_family` объединяются через OR. Одна дата учитывается ровно один раз, даже если в неё одновременно положительны несколько горизонтов или порогов. Частота считается по календарной длине общего размеченного периода; `OK` означает в среднем от 2 до 3 уникальных сигналов в неделю.


In [6]:
target_family_frequency = evaluate_target_family_frequency(
    update_rows,
    target_registry=target_registry,
    min_signals_per_week=1.0,
    max_signals_per_week=5.0,
)

target_family_frequency[[
    "currency", "scenario", "target_family",
    "configuration_count", "observations",
    "raw_positive_count_with_duplicates",
    "unique_signal_count", "overlap_removed_count",
    "unique_target_share", "min_required_target_share",
    "max_required_target_share", "unique_signals_per_week",
    "target_frequency_status",
]]


,currency,scenario,target_family,configuration_count,observations,raw_positive_count_with_duplicates,unique_signal_count,overlap_removed_count,unique_target_share,min_required_target_share,max_required_target_share,unique_signals_per_week,target_frequency_status
0,AMD,GOOD_NOW,G0,5,1621,892,444,448,0.273905,0.211333,1.056667,1.296080,OK
1,KGS,GOOD_NOW,G0,5,1621,899,448,451,0.276373,0.211333,1.056667,1.307756,OK
2,KZT,GOOD_NOW,G0,5,1621,963,472,491,0.291178,0.211333,1.056667,1.377815,OK
3,TJS,GOOD_NOW,G0,5,1621,890,446,444,0.275139,0.211333,1.056667,1.301918,OK
4,UZS,GOOD_NOW,G0,5,1621,893,438,455,0.270204,0.211333,1.056667,1.278565,OK
5,AMD,GOOD_NOW,G1,15,1629,14173,1468,12705,0.901166,0.212049,1.060247,4.249793,OK
6,KGS,GOOD_NOW,G1,15,1629,13906,1471,12435,0.903008,0.212049,1.060247,4.258478,OK
7,KZT,GOOD_NOW,G1,15,1629,13727,1466,12261,0.899939,0.212049,1.060247,4.244003,OK
8,TJS,GOOD_NOW,G1,15,1629,14368,1484,12884,0.910988,0.212049,1.060247,4.296112,OK
9,UZS,GOOD_NOW,G1,15,1629,13779,1474,12305,0.904850,0.212049,1.060247,4.267163,OK


In [7]:
target_family_frequency[['currency', 'scenario', 'target_family', 'unique_signals_per_week']]

,currency,scenario,target_family,unique_signals_per_week
0,AMD,GOOD_NOW,G0,1.296080
1,KGS,GOOD_NOW,G0,1.307756
2,KZT,GOOD_NOW,G0,1.377815
3,TJS,GOOD_NOW,G0,1.301918
4,UZS,GOOD_NOW,G0,1.278565
5,AMD,GOOD_NOW,G1,4.249793
6,KGS,GOOD_NOW,G1,4.258478
7,KZT,GOOD_NOW,G1,4.244003
8,TJS,GOOD_NOW,G1,4.296112
9,UZS,GOOD_NOW,G1,4.267163


## 4. Пространства правил

Сначала задаются простые пространства правил: тип индикатора, окно и порог. Затем явно строятся пространства кандидатов: одиночные индикаторы и все попарные сочетания разных типов через `AND` и `OR`. Порог и окно выбираются только на train каждого WF-фолда.


In [8]:
def rules_for_windows(indicator_type, feature_template, windows, operator, thresholds):
    rules = []
    for window in windows:
        rules.extend(create_indicator_rules(
            indicator_type=indicator_type,
            feature=feature_template.format(window=window),
            operator=operator,
            thresholds=thresholds,
        ))
    return rules

RULE_SPACES = {
    "level_low": rules_for_windows(
        "level_low", "percentile_{window}d", (30, 90, 180), "le", (0.05, 0.10, 0.15, 0.20)
    ),
    "near_low": rules_for_windows(
        "near_low", "distance_from_low_{window}d_bps", (30, 90, 180), "le", (25, 50, 100)
    ),
    "momentum_down": rules_for_windows(
        "momentum_down", "return_{window}d_bps", (1, 3, 5, 10, 20), "le", (-200, -100, -50, 0)
    ),
    "momentum_up": rules_for_windows(
        "momentum_up", "return_{window}d_bps", (1, 3, 5, 10, 20), "ge", (0, 50, 100, 200)
    ),
    "down_streak": create_indicator_rules(
        "down_streak", "consecutive_down", "ge", (2, 3, 4, 5)
    ),
    "up_streak": create_indicator_rules(
        "up_streak", "consecutive_up", "ge", (2, 3, 4, 5)
    ),
    "trend_down": rules_for_windows(
        "trend_down", "slope_{window}d_bps_per_day", (3, 5, 10), "le", (-50, 0)
    ),
    "trend_up": rules_for_windows(
        "trend_up", "slope_{window}d_bps_per_day", (3, 5, 10), "ge", (0, 50)
    ),
    "high_volatility": create_indicator_rules(
        "high_volatility", "volatility_ratio_7d_30d", "ge", (0.8, 1.0, 1.2, 1.5)
    ),
}

# Один элемент INDICATOR_SPACES — одна архитектура индикатора.
# Внутри лежат варианты её параметров, выбираемые на train.
INDICATOR_SPACES = {
    name: [IndicatorCandidate((rule,)) for rule in rules]
    for name, rules in RULE_SPACES.items()
}

for left_name, right_name in combinations(RULE_SPACES, 2):
    for logic in ("AND", "OR"):
        space_name = f"{left_name}__{logic}__{right_name}"
        INDICATOR_SPACES[space_name] = [
            IndicatorCandidate((left_rule, right_rule), logic=logic)
            for left_rule, right_rule in product(
                RULE_SPACES[left_name], RULE_SPACES[right_name]
            )
        ]

pd.DataFrame([
    {
        "indicator": name,
        "logic": candidates[0].logic,
        "parameter_variants": len(candidates),
    }
    for name, candidates in INDICATOR_SPACES.items()
]).sort_values(["logic", "indicator"]).reset_index(drop=True)


,indicator,logic,parameter_variants
0,down_streak__AND__high_volatility,AND,16
1,down_streak__AND__trend_down,AND,24
2,down_streak__AND__trend_up,AND,24
3,down_streak__AND__up_streak,AND,16
4,level_low__AND__down_streak,AND,48
...,...,...,...
76,momentum_up,SINGLE,20
77,near_low,SINGLE,9
78,trend_down,SINGLE,6
79,trend_up,SINGLE,6


## 5. Walk-forward оптимизация

Для каждой валюты один раз вычисляется общая булева матрица всех простых правил и их комбинаций. Затем одни и те же матрица и WF-разбиения переиспользуются для всех targets, horizons и indicator spaces. Методология не меняется:

```text
train → оставляем кандидатов с ≥ 2 сигналов/неделю и максимизируем lift
следующие test_months месяцев → test выбранного кандидата
расширяем train → снова выбираем параметры → следующий test
```

Для каждого кандидата `random_precision` равна доле таргета на том же временном отрезке, а `lift = signal_precision / random_precision`. Итоговый `oos_lift` и средняя частота считаются один раз по всем test-folds вместе. Победитель конкретного `currency × target × horizon` должен выполнять ограничение частоты на каждом train-fold, на полном fit и в среднем OOS; затем выбирается максимальный pooled OOS lift. Precision сохраняется только как диагностический числитель lift. Discovery заканчивается за максимальный target horizon до `FINAL_BACKTEST_START`, поэтому outcomes финального holdout не участвуют в выборе.


In [9]:
leaderboard_tables = []
fold_tables = []
ensemble_tables = []
rule_signal_tables = []
fitted_candidates = {}

# Purge max horizon: discovery не использует outcomes из final holdout.
discovery_end_exclusive = (
    FINAL_BACKTEST_START - pd.Timedelta(days=max(DEFAULT_HORIZONS))
)
discovery_rows = update_rows.loc[
    update_rows["available_at"].lt(discovery_end_exclusive)
].copy()

for currency in RECIPIENT_CURRENCIES:
    currency_data = discovery_rows.loc[
        discovery_rows["currency"].eq(currency)
    ].reset_index(drop=True)
    summaries, folds, currency_fitted, ensembles, rule_signals = optimize_indicator_library(
        currency_data,
        target_registry=target_registry,
        indicator_spaces=INDICATOR_SPACES,
        first_test_date=FIRST_TEST_DATE,
        test_months_options=TEST_MONTHS_OPTIONS,
        min_signals_per_week=INDICATOR_MIN_SIGNALS_PER_WEEK,
        ensemble_lift_thresholds=ENSEMBLE_LIFT_THRESHOLDS,
    )
    leaderboard_tables.append(summaries.assign(currency=currency))
    fold_tables.append(folds.assign(currency=currency))
    ensemble_tables.append(ensembles.assign(currency=currency))
    rule_signal_tables.append(rule_signals.assign(currency=currency))
    fitted_candidates.update({
        (currency, *key): candidate
        for key, candidate in currency_fitted.items()
    })

indicator_leaderboard = pd.concat(leaderboard_tables, ignore_index=True)
indicator_leaderboard = indicator_leaderboard.sort_values(
    ["currency", "target", "horizon", "oos_lift", "indicator", "test_months"],
    ascending=[True, True, True, False, True, True],
).reset_index(drop=True)
indicator_leaderboard["eligible_for_selection"] = (
    indicator_leaderboard["fitted_frequency_constraint_met"]
    & indicator_leaderboard["oos_frequency_constraint_met"]
    & indicator_leaderboard["frequency_eligible_folds"].eq(
        indicator_leaderboard["folds"]
    )
)
indicator_leaderboard["rank"] = pd.Series(
    pd.NA, index=indicator_leaderboard.index, dtype="Int64"
)
eligible_rows = indicator_leaderboard["eligible_for_selection"]
indicator_leaderboard.loc[eligible_rows, "rank"] = (
    indicator_leaderboard.loc[eligible_rows]
    .groupby(["currency", "target", "horizon"])
    .cumcount()
    .add(1)
)
indicator_fold_results = pd.concat(fold_tables, ignore_index=True)
indicator_ensemble_summary = pd.concat(ensemble_tables, ignore_index=True)
indicator_ensemble_oos_signals = pd.concat(rule_signal_tables, ignore_index=True)

best_indicators = indicator_leaderboard.loc[
    indicator_leaderboard["rank"].eq(1),
    [
        "currency", "scenario", "target_family", "target", "horizon",
        "indicator", "test_months", "oos_lift",
        "oos_precision", "oos_random_precision",
        "oos_signals_per_week", "test_predicted_positive_count",
        "test_true_positive", "folds", "frequency_eligible_folds",
        "fitted_candidate", "fitted_logic", "fitted_rule_count",
    ],
].reset_index(drop=True)

# В памяти оставляем только один финально обученный кандидат на конфигурацию.
best_fitted_indicators = {
    (row.currency, row.target, int(row.horizon)): fitted_candidates[(
        row.currency, row.target, int(row.horizon), row.indicator, int(row.test_months)
    )]
    for row in best_indicators.itertuples(index=False)
}
del fitted_candidates
best_indicators


,currency,scenario,target_family,target,horizon,indicator,test_months,oos_lift,oos_precision,oos_random_precision,oos_signals_per_week,test_predicted_positive_count,test_true_positive,folds,frequency_eligible_folds,fitted_candidate,fitted_logic,fitted_rule_count
0,AMD,GOOD_NOW,G0,target_g0_exact_min_h10d,10,trend_down,3,2.340836,0.096463,0.041209,2.053774,311,30,12,12,trend_down__slope_5d_bps_per_day__le_0,SINGLE,1
1,AMD,GOOD_NOW,G0,target_g0_exact_min_h1d,1,momentum_down,3,2.212766,0.534954,0.241758,2.172642,329,176,12,12,momentum_down__return_1d_bps__le_0,SINGLE,1
2,AMD,GOOD_NOW,G0,target_g0_exact_min_h20d,20,momentum_down__OR__momentum_up,3,2.371336,0.045603,0.019231,2.027358,307,14,12,12,momentum_down__return_10d_bps__le_m50__OR__mom...,OR,2
3,AMD,GOOD_NOW,G0,target_g0_exact_min_h3d,3,momentum_down,6,2.253870,0.263158,0.116758,2.125000,323,85,6,6,momentum_down__return_3d_bps__le_0,SINGLE,1
4,AMD,GOOD_NOW,G0,target_g0_exact_min_h5d,5,level_low__OR__trend_down,3,2.289308,0.176101,0.076923,2.100000,318,56,12,12,level_low__percentile_30d__le_0p05__OR__trend_...,OR,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,UZS,WINDOW_CLOSING,W1,target_w1_lowpct_0p15_deterioration_75bps_h10d,10,level_low__OR__high_volatility,12,2.379085,0.192810,0.081044,2.009381,306,59,3,3,level_low__percentile_90d__le_0p15__OR__high_v...,OR,2
146,UZS,WINDOW_CLOSING,W1,target_w1_lowpct_0p15_deterioration_75bps_h1d,1,level_low__OR__trend_down,6,2.182545,0.056962,0.026099,2.078947,316,18,6,6,level_low__percentile_90d__le_0p15__OR__trend_...,OR,2
147,UZS,WINDOW_CLOSING,W1,target_w1_lowpct_0p15_deterioration_75bps_h20d,20,level_low__OR__high_volatility,12,2.379085,0.209150,0.087912,2.009381,306,64,3,3,level_low__percentile_90d__le_0p15__OR__high_v...,OR,2
148,UZS,WINDOW_CLOSING,W1,target_w1_lowpct_0p15_deterioration_75bps_h3d,3,level_low__OR__high_volatility,12,2.379085,0.124183,0.052198,2.009381,306,38,3,3,level_low__percentile_90d__le_0p15__OR__high_v...,OR,2


In [10]:
mask = (best_indicators['scenario'] == 'WINDOW_CLOSING') & (best_indicators['target_family'] == 'W1')
best_indicators.loc[mask, ['currency', 'scenario', 'target_family', 'target', 'horizon', 'oos_lift']]

,currency,scenario,target_family,target,horizon,oos_lift
25,AMD,WINDOW_CLOSING,W1,target_w1_lowpct_0p15_deterioration_75bps_h10d,10,2.296201
26,AMD,WINDOW_CLOSING,W1,target_w1_lowpct_0p15_deterioration_75bps_h1d,1,2.371336
27,AMD,WINDOW_CLOSING,W1,target_w1_lowpct_0p15_deterioration_75bps_h20d,20,2.402640
28,AMD,WINDOW_CLOSING,W1,target_w1_lowpct_0p15_deterioration_75bps_h3d,3,2.402640
29,AMD,WINDOW_CLOSING,W1,target_w1_lowpct_0p15_deterioration_75bps_h5d,5,2.325879
55,KGS,WINDOW_CLOSING,W1,target_w1_lowpct_0p15_deterioration_75bps_h10d,10,2.311111
56,KGS,WINDOW_CLOSING,W1,target_w1_lowpct_0p15_deterioration_75bps_h1d,1,2.318471
57,KGS,WINDOW_CLOSING,W1,target_w1_lowpct_0p15_deterioration_75bps_h20d,20,2.296530
58,KGS,WINDOW_CLOSING,W1,target_w1_lowpct_0p15_deterioration_75bps_h3d,3,2.379085
59,KGS,WINDOW_CLOSING,W1,target_w1_lowpct_0p15_deterioration_75bps_h5d,5,2.339045


## 6. Независимый backtest замороженных G0/W1-индикаторов

Берутся только победители семейств `G0` и `W1`. Их concrete rules, thresholds и logic полностью фиксируются по discovery-периоду и без нового подбора применяются к данным начиная с `FINAL_BACKTEST_START`. Выбранный `test_months` задаёт частоту контрольных срезов backtest, но не изменяет параметры правила. Итоговая таблица считает фактическое число OOS-сигналов, pooled lift и среднее число сигналов за календарную неделю.


In [11]:
(
    final_indicator_backtest_summary,
    final_indicator_backtest_folds,
    final_indicator_backtest_signals,
) = backtest_fixed_indicators(
    update_rows,
    selected_indicators=best_indicators,
    fitted_indicators=best_fitted_indicators,
    backtest_start=FINAL_BACKTEST_START,
    target_families=FINAL_BACKTEST_TARGET_FAMILIES,
)

final_indicator_backtest_summary[[
    "currency", "scenario", "target_family", "target", "horizon",
    "indicator", "fixed_candidate", "fixed_logic",
    "rebalance_months", "folds", "test_observations",
    "test_signal_count", "test_true_positive", "test_false_positive",
    "test_signal_precision", "test_random_precision", "test_lift",
    "test_calendar_weeks", "test_signals_per_week",
]]


,currency,scenario,target_family,target,horizon,indicator,fixed_candidate,fixed_logic,rebalance_months,folds,test_observations,test_signal_count,test_true_positive,test_false_positive,test_signal_precision,test_random_precision,test_lift,test_calendar_weeks,test_signals_per_week
0,AMD,GOOD_NOW,G0,target_g0_exact_min_h1d,1,momentum_down,momentum_down__return_1d_bps__le_0,SINGLE,3,7,409,203,116,87,0.571429,0.283619,2.014778,87.285714,2.325696
1,AMD,GOOD_NOW,G0,target_g0_exact_min_h3d,3,momentum_down,momentum_down__return_3d_bps__le_0,SINGLE,6,4,407,206,53,153,0.257282,0.130221,1.975728,86.714286,2.375618
2,AMD,GOOD_NOW,G0,target_g0_exact_min_h5d,5,level_low__OR__trend_down,level_low__percentile_30d__le_0p05__OR__trend_...,OR,3,7,407,209,40,169,0.191388,0.098280,1.947368,86.714286,2.410214
3,AMD,GOOD_NOW,G0,target_g0_exact_min_h10d,10,trend_down,trend_down__slope_5d_bps_per_day__le_0,SINGLE,3,7,402,207,20,187,0.096618,0.049751,1.942029,85.714286,2.415000
4,AMD,GOOD_NOW,G0,target_g0_exact_min_h20d,20,momentum_down__OR__momentum_up,momentum_down__return_10d_bps__le_m50__OR__mom...,OR,3,7,396,196,9,187,0.045918,0.022727,2.020408,84.571429,2.317568
5,KGS,GOOD_NOW,G0,target_g0_exact_min_h1d,1,momentum_down,momentum_down__return_1d_bps__le_0,SINGLE,3,7,409,223,118,105,0.529148,0.288509,1.834081,87.285714,2.554828
6,KGS,GOOD_NOW,G0,target_g0_exact_min_h3d,3,momentum_down,momentum_down__return_3d_bps__le_0,SINGLE,3,7,407,216,57,159,0.263889,0.140049,1.884259,86.714286,2.490939
7,KGS,GOOD_NOW,G0,target_g0_exact_min_h5d,5,momentum_down,momentum_down__return_3d_bps__le_0,SINGLE,3,7,407,216,38,178,0.175926,0.093366,1.884259,86.714286,2.490939
8,KGS,GOOD_NOW,G0,target_g0_exact_min_h10d,10,near_low__OR__trend_down,near_low__distance_from_low_180d_bps__le_25__O...,OR,3,7,402,223,18,205,0.080717,0.044776,1.802691,85.714286,2.601667
9,KGS,GOOD_NOW,G0,target_g0_exact_min_h20d,20,momentum_down__OR__up_streak,momentum_down__return_20d_bps__le_m50__OR__up_...,OR,12,2,396,211,7,204,0.033175,0.017677,1.876777,84.571429,2.494932
